# COMPASS multivariate text models

Elastic-net Cox and XGBoost on pooled clinical-note embeddings, predicting
time-to-platinum from ADT start. Runs after `01_preprocessing.ipynb` has built the
standard prediction inputs, and after `cp.build_text_embedding_inputs(run)` has built
the matched text inputs from them (Stage 0 below).

Note embeddings come from the sibling `clinical_text_embedding_project`: three note
types (Clinician, Imaging, Pathology) x 768 dimensions, pooled per patient with
`time_decay_mean` over notes **strictly before the landmark**. The pooling helper is
the embedding project's own (`survival/preprocessing.py:generate_survival_embedding_df`),
which enforces the same landmark contract `build_prediction_inputs.py` does, so the
leakage control is shared rather than reimplemented.

## Feature arms

| feature_set | predictors | reads |
|---|---|---|
| `labs` | summary labs + age (the reference arm) | `text_embedding/labs/` |
| `text` | 2304 pooled embedding dims + age | `text_embedding/text/` |
| `labs_text` | both | `text_embedding/labs_text/` |

**All three arms are fit on one shared patient set**, not on their natural cohorts.
The embedding project requires all three pre-anchor note modalities per patient, so
the text cohort is a strict subset of the ADT cohort. The `labs` arm is therefore
*refit here on that matched subset* rather than compared against
`03_multivariate.ipynb`'s numbers — otherwise the arm difference would be confounded
with the cohort difference. `delta_c_index` in the summary table is the paired
contrast that this matching makes interpretable.

### To predict time-to-platinum on the full ADT cohort

```python
ARMS       = ["adt"]
ENDPOINTS  = ("platinum",)
COHORTS    = ("all",)          # no MRN restriction
EXCLUSIONS = ("none",)
BUILD_TEXT_INPUTS = True       # once; then set back to False
```

In [ ]:
from pathlib import Path

ARMS = ["adt"]

# Time-to-platinum on the FULL ADT cohort, matching 03/03b's headline cell.
ENDPOINTS = ("platinum",)          # was: ("platinum", "nepc")
COHORTS = ("all",)                 # full arm cohort, no MRN restriction
# Orthogonal to COHORTS: "none" keeps every patient, "pre_adt_castrate"
# drops those with a castrate testosterone (<50 ng/dL) before ADT start.
EXCLUSIONS = ("none",)

# Stage 0: build the three matched arms under inputs_dir/text_embedding/.
# This reads the standard landmark frames, joins ADT-relative note times, pools
# embeddings per landmark, and takes the complete-case intersection. It is a
# one-time build per prediction-input tree -- set back to False afterwards, since
# the fits below read its output and do not depend on it re-running.
#
# PREREQUISITE: the clinical_text_embedding_project checkout (for its pooling
# code) and its embedding data:
#   full_clinical_notes_embeddings_as_array.npy.zst
#   full_clinical_notes_embeddings_metadata.parquet
# The checkout is found automatically in the usual neighboring layouts; override
# with CTEP_REPO_PATH if it lives elsewhere. The DATA files are resolved by that
# project's own config.py (NOTES_PATH, under CTEP_DATA_PATH), which defaults to
# the cluster root -- so like the rest of COMPASS, Stage 0 runs on the cluster,
# not on a local checkout. The next cell prints both paths and whether they
# resolve, before anything expensive runs.
BUILD_TEXT_INPUTS = False

# COST: elastic-net over 2304 dense columns x 5 folds is far more expensive than
# the lab arms (tens of columns), and XGBoost on 2304 dense features is also
# substantially slower. To smoke-test before committing to the full grid, narrow
# to one landmark after the next cell:
#   RUNS[0]["landmarks"] = [0]
# That is the cheapest run that answers the gating question -- whether text
# carries signal for this endpoint on the matched cohort.

OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.FORCE_RERUN = OVERWRITE

RUNS = cp.make_endpoint_runs(
    ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS,
)
print(f"feature arms: {list(cp.TEXT_FEATURE_SETS)}")
print(f"models: {sorted({m for m, _, _ in cp.TEXT_TASK_SPECS})}")
print(f"\n{len(RUNS)} run(s): {[r['label'] for r in RUNS]}")

## Stage 0 — build the matched text inputs

Skipped unless `BUILD_TEXT_INPUTS` was set. The builder prints per-landmark
retention (how many of the landmark cohort survive the three-modality
complete-case requirement) and records `n_patients_by_landmark` and
`split_sizes_by_landmark` in each arm's `build_manifest.json`. Read that retention
before reading the model results: if attrition is severe, the matched `labs`
comparator loses power and every `delta_c_index` below gets correspondingly
noisier.

AUC horizons are recomputed on the matched cohort rather than inherited, because
the subset's event-time quantiles differ from the full cohort's.

In [ ]:
# Where the text side resolves to, and whether it is actually reachable from
# here. Printed before the build so a wrong path shows up now rather than as a
# failure partway through pooling.
import build_text_embedding_inputs as bte

ctep_repo = bte.CLINICAL_EMBEDDINGS_REPO
print(f"embedding repo:  {ctep_repo}")
print(f"  readable:      {(ctep_repo / 'anchors.py').exists()}")
try:
    _, notes_path = bte._import_embedding_helpers()
    notes_dir = Path(notes_path)
    print(f"embedding data:  {notes_dir}")
    print(f"  readable:      {notes_dir.exists()}")
    if not notes_dir.exists():
        print(
            "\n[warn] The embedding data is not reachable from this machine.\n"
            "       Stage 0 reads it from the cluster path in the embedding\n"
            "       project's config.py (override with CTEP_DATA_PATH). Run this\n"
            "       notebook where that data lives, as with the rest of COMPASS."
        )
except (FileNotFoundError, ImportError) as exc:
    print(f"\n[warn] Could not load the embedding project: {exc}")

if BUILD_TEXT_INPUTS:
    for run in RUNS:
        cp.build_text_embedding_inputs(run)
else:
    print("\nBUILD_TEXT_INPUTS is False -- using existing text_embedding inputs.")

# Check what the fits will actually read, now, rather than failing inside a fit.
# Each arm is a self-contained inputs tree with its own aggregated landmark frame.
# Importing compass_pipeline above put data_preprocessing/ on sys.path, which is where
# these filename helpers live. The builder and the fits must agree on them, so they are
# imported rather than restated as a literal pattern here.
from build_prediction_inputs import BUILD_MANIFEST_FILENAME, aggregated_filename

for run in RUNS:
    missing = [
        str(run["inputs_dir"] / cp.TEXT_EMBEDDING_DIRNAME / fs / aggregated_filename(d))
        for fs in cp.TEXT_FEATURE_SETS
        for d in run["landmarks"]
        if not (
            run["inputs_dir"] / cp.TEXT_EMBEDDING_DIRNAME / fs / aggregated_filename(d)
        ).exists()
    ]
    if missing:
        n_expected = len(cp.TEXT_FEATURE_SETS) * len(run["landmarks"])
        print(
            f"\n[warn] {run['label']}: text inputs are missing "
            f"({len(missing)} of {n_expected} arm x landmark frames), e.g.\n"
            f"        {missing[0]}\n"
            "        Set BUILD_TEXT_INPUTS = True in the configuration cell and "
            "re-run it, or call\n"
            "        cp.build_text_embedding_inputs(run) directly."
        )
    else:
        print(f"[ok  ] {run['label']}: all text inputs present.")

## Run the three feature arms

Elastic-net Cox and XGBoost x {`labs`, `text`, `labs_text`} at every landmark. Set
`OVERWRITE = True` in the configuration cell to refit and replace existing outputs.
With `False`, tasks whose metrics file already exists are skipped, so an interrupted
run resumes where it stopped. Layout:
`local_runs_<arm>/text_embedding/<model>/landmark_{0,90,180}/<feature_set>/`.

Each arm writes a distinct metrics filename in its own directory, so no arm can
overwrite another and each resumes independently — the same convention 03b uses for
its two DeepHit arms.

Two details make the `text` arm fit correctly, both handled in
`cox_aggregated.prepare_landmark_context`. The embedding columns are declared as
`always_include_feature_cols` so the canonical-lab gate cannot drop them: column
names like `CLINICIAN_EMBEDDING_42` carry no `__`, and `parse_feature_name` would
otherwise treat them as unrecognized labs and silently drop all 2304 — leaving an
age-only model that still reports a plausible C-index. The genomic prevalence floor
is also left unset for these arms, since a 2.5% prevalence threshold is meaningless
against dense continuous dimensions.

In [ ]:
for run in RUNS:
    cp.run_multivariate_text(run)

## Summary tables

Per-run C-index / mean AUC(t) / integrated Brier for every (model, landmark,
feature_set), plus `delta_c_index`: the arm's C-index minus the `labs` arm's at the
same (model, landmark). NaN for the `labs` rows themselves, and wherever the `labs`
reference is missing, so a partial run shows gaps rather than a spurious zero.

Read `labs_text` vs `labs` as the question of record — the incremental value of
clinical text over the summary labs already available. `text` vs `labs` is the
secondary question of whether text alone is competitive.

In [ ]:
summary_dfs = {cp.run_key(run): cp.summarize_text_outputs(run) for run in RUNS}
for label, df in summary_dfs.items():
    print(f"=== {label} ===")
    display(df)

In [ ]:
import pandas as pd

combined_text_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_text_summary_df

## Cohort retention

The attrition the whole comparison rests on, read back from the build manifests
rather than recomputed. `n_patients` is the matched cohort at that landmark; the
three arms share it by construction, so a single row per landmark describes all of
them.

In [ ]:
import json

import pandas as pd

# Imported here as well as in Stage 0 so this cell can be re-run on its own.
from build_prediction_inputs import BUILD_MANIFEST_FILENAME

retention_rows = []
for run in RUNS:
    for landmark_day in run["landmarks"]:
        base_path = run["inputs_dir"] / BUILD_MANIFEST_FILENAME
        arm_path = (
            run["inputs_dir"] / cp.TEXT_EMBEDDING_DIRNAME
            / cp.TEXT_FEATURE_SETS[0] / BUILD_MANIFEST_FILENAME
        )
        if not arm_path.exists():
            continue
        arm_manifest = json.loads(arm_path.read_text())
        n_matched = arm_manifest.get("n_patients_by_landmark", {}).get(str(landmark_day))
        n_base = None
        if base_path.exists():
            n_base = json.loads(base_path.read_text()).get(
                "n_patients_by_landmark", {}
            ).get(str(landmark_day))
        retention_rows.append(
            {
                "run": run["label"],
                "landmark": landmark_day,
                "n_base_cohort": n_base,
                "n_text_matched": n_matched,
                "retained": (
                    round(n_matched / n_base, 3)
                    if n_base and n_matched is not None
                    else None
                ),
                "splits": arm_manifest.get("split_sizes_by_landmark", {}).get(
                    str(landmark_day)
                ),
            }
        )

if not retention_rows:
    print("No text build manifest found -- run Stage 0 first.")
else:
    display(pd.DataFrame(retention_rows))